## 1. Char RNN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

### 1.1 훈련 데이터 전처리

In [2]:
string = 'pineapple!'
input_str = string[:-1]  # 'pineapple'
label_str = string[1:]  # 'ineapple!'

char_vocab = sorted(list(set(string)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 7


In [3]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = vocab_size # 출력의 크기는 문자 집합의 크기
learning_rate = 0.1

In [4]:
# 문자에 고유한 정수 인덱스 부여: 인코딩
char_to_index = dict((c, i) for i, c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'i': 3, 'l': 4, 'n': 5, 'p': 6}


In [5]:
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'i', 4: 'l', 5: 'n', 6: 'p'}


In [6]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[6, 3, 5, 2, 1, 6, 6, 4, 2]
[3, 5, 2, 1, 6, 6, 4, 2, 0]


In [7]:
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[6, 3, 5, 2, 1, 6, 6, 4, 2]]
[[3, 5, 2, 1, 6, 6, 4, 2, 0]]


In [8]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.]])]


In [9]:
X = torch.FloatTensor(np.array(x_one_hot))
Y = torch.LongTensor(np.array(y_data))

In [10]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 9, 7])
레이블의 크기 : torch.Size([1, 9])


### 1.2 모델 구현

In [11]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [12]:
net = Net(input_size, hidden_size, output_size)

In [13]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([1, 9, 7])


In [14]:
print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환

torch.Size([9, 7])


In [15]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 9])
torch.Size([9])


In [16]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [17]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  2.1412878036499023 prediction:  [[5 4 4 4 4 5 5 5 4]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  nllllnnnl
1 loss:  1.871193528175354 prediction:  [[6 6 6 6 6 6 6 6 6]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  ppppppppp
2 loss:  1.6965492963790894 prediction:  [[2 6 6 6 6 6 6 6 6]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  epppppppp
3 loss:  1.5540739297866821 prediction:  [[2 2 2 2 6 6 6 2 6]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  eeeepppep
4 loss:  1.4207924604415894 prediction:  [[2 2 2 2 6 6 6 2 0]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  eeeepppe!
5 loss:  1.283384919166565 prediction:  [[3 5 2 0 6 6 6 2 0]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  ine!pppe!
6 loss:  1.1494708061218262 prediction:  [[3 5 2 0 6 6 6 2 0]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0]] prediction str:  ine!pppe!
7 loss:  1.0194878578186035 prediction:  [[3 5 2 0 6 6 6 2 0]] true Y:  [[3, 5, 2, 1, 6, 6, 4, 2, 0

## 2. 더 많은 데이터로 학습한 문자 단위 RNN(Char RNN)

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

### 2.1 훈련 데이터 전처리

In [19]:
sentence = """
if you want to build a ship, don't drum up people together to 
collect wood and don't assign them tasks and work, but rather 
teach them to long for the endless immensity of the sea.
"""

print(sentence)


if you want to build a ship, don't drum up people together to 
collect wood and don't assign them tasks and work, but rather 
teach them to long for the endless immensity of the sea.



In [20]:
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩

In [21]:
print(char_dic) # 공백과 줄바꿈도 여기서는 하나의 원소

{'b': 0, 'e': 1, 's': 2, 'r': 3, 'w': 4, 'n': 5, 'g': 6, ' ': 7, '.': 8, 'h': 9, 'k': 10, 'm': 11, 'p': 12, 't': 13, ',': 14, 'o': 15, "'": 16, '\n': 17, 'a': 18, 'l': 19, 'i': 20, 'f': 21, 'y': 22, 'u': 23, 'd': 24, 'c': 25}


In [22]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 26


In [23]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

In [24]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 
if you wa -> if you wan
1 if you wan -> f you want
2 f you want ->  you want 
3  you want  -> you want t
4 you want t -> ou want to
5 ou want to -> u want to 
6 u want to  ->  want to b
7  want to b -> want to bu
8 want to bu -> ant to bui
9 ant to bui -> nt to buil
10 nt to buil -> t to build
11 t to build ->  to build 
12  to build  -> to build a
13 to build a -> o build a 
14 o build a  ->  build a s
15  build a s -> build a sh
16 build a sh -> uild a shi
17 uild a shi -> ild a ship
18 ild a ship -> ld a ship,
19 ld a ship, -> d a ship, 
20 d a ship,  ->  a ship, d
21  a ship, d -> a ship, do
22 a ship, do ->  ship, don
23  ship, don -> ship, don'
24 ship, don' -> hip, don't
25 hip, don't -> ip, don't 
26 ip, don't  -> p, don't d
27 p, don't d -> , don't dr
28 , don't dr ->  don't dru
29  don't dru -> don't drum
30 don't drum -> on't drum 
31 on't drum  -> n't drum u
32 n't drum u -> 't drum up
33 't drum up -> t drum up 
34 t drum up  ->  drum up p
35  drum up p -> drum up pe
36

In [25]:
print(x_data[0])
print(y_data[0])

[17, 20, 21, 7, 22, 15, 23, 7, 4, 18]
[20, 21, 7, 22, 15, 23, 7, 4, 18, 5]


In [26]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(np.array(x_one_hot))
Y = torch.LongTensor(np.array(y_data))

In [27]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))
print(f'첫번째 훈련 데이터: {X[0]}')
print(f'첫번째 레이블: {Y[0]}')

훈련 데이터의 크기 : torch.Size([174, 10, 26])
레이블의 크기 : torch.Size([174, 10])
첫번째 훈련 데이터: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,

### 2.2 모델 구현

In [28]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [29]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.

In [30]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [31]:
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([174, 10, 26])


In [32]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.

torch.Size([1740, 26])


In [33]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([174, 10])
torch.Size([1740])


In [34]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

oooooooooooooooooooooyooooooooooooooooooooooooooooyoooooooooooooiooooiooooooooooooooooooooooooooooooooyoyooooooooooooooooooooooyoooooooooooooooooyooooooooooooooooooooooyoooooooooooooo
        sb     b b b                                            b b     k              b                 b            sb         b b                         b                         
oo'boldbolollllllllolllllollllllllllollallollolllolllllllollllllllllllllllllolllllllllllolallolllllolollllllllllllllalllllllolllllllllllllllllllllllllolollllllllllloldllllllalolllllll
            t  to  o to o  n to    to n  o  n  n  to    e  to  o  e    to       eo    to  o  t    to  n to  no    to  t  on  no e  e     to  o    o  no n o  e  n o     e  to no o    n
t  to   on  tonoue o douo ou ao ouodoeoyoo ouo oe t  ouoeo om ooo oe   dono ooeououo  do o'o o e  o   doto odoeo  o e uo oeo oi    n'oeo ou tu t ooe 'oeodouoeo o'o   e doeo  t e tono'
t  ton uontutontuo o touo ogououououououtu tuouou to ouoeouou tun tuouutououoo  